# CAM C04 NICE Pipeline — Google Colab Notebook

Colab adaptation of the `CAM_C04_NICE_Project` integrated SNOMED retrieval pipeline.  
The original source files are **not modified** by this notebook.

**Before running:**
1. Run the Setup cells (Section 1) to install packages and mount your Drive.
2. Edit **Section 2 · User Settings** to point at your own resource paths.
3. Run all cells top-to-bottom.

---
## Section 1 · Setup
Install dependencies and connect to Google Drive.

In [129]:
!pip install -q chromadb langchain-ollama sentence-transformers rank-bm25 pandas numpy

In [130]:
# Install zstd dependency and then Ollama
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
!nohup ollama serve > ollama.log 2>&1 &

import time
time.sleep(5) # Give the server time to start

# Pull the specific model used in settings
!ollama pull llama3.1

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [131]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [132]:
import json
import logging
import math
import re
import uuid
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
)
logger = logging.getLogger('ColabPipeline')

---
## Section 2 · User Settings

> **Edit these variables before running the pipeline.**  
> The paths shown are *example placeholders only* — they will not work without being updated.  
> Replace them with the actual locations of your resources in Google Drive  
> (MyDrive, a Shared Drive, or any mounted path).

In [133]:
# ── Resource Paths ────────────────────────────────────────────────────────────
# Main concept table, hierarchy edges, new Chroma store, and local model cache.
# Update these to match where your shared resources actually live in Drive.

SNOMED_PATH    = '/content/drive/MyDrive/NICE/snomed_master_v4.csv'
EDGE_PATH      = '/content/drive/MyDrive/NICE/snomed_parent_child_edges_clean.csv'
CHROMA_DIR     = '/content/drive/MyDrive/NICE/chroma_db_v4'
EMBEDDINGS_DIR = '/content/embeddings_cache'

# ── Models ────────────────────────────────────────────────────────────────────
# Same embedding model as before, but used to build a fresh v4 vector index.

EMBEDDING_MODEL_NAME = 'BAAI/bge-small-en'
LLM_MODEL            = 'llama3.1'

# ── Runtime ───────────────────────────────────────────────────────────────────
# Simple demo query and retrieval size for testing.

DEMO_QUERY = 'Obesity, diabetes mellitus, and hypertension'
TOP_K      = 20

In [134]:
import os
from pathlib import Path

# Check whether the required project files and directories are available.
# Note: the new Chroma directory may not exist yet if you have not rebuilt the collection.

paths_to_check = {
    'SNOMED CSV': SNOMED_PATH,
    'Hierarchy Edge CSV': EDGE_PATH,
    'Chroma DB': CHROMA_DIR
}

for name, path in paths_to_check.items():
    p = Path(path)
    if p.exists():
        print(f'✅ {name} found at: {path}')
    else:
        print(f'❌ {name} NOT found at: {path}')

✅ SNOMED CSV found at: /content/drive/MyDrive/NICE/snomed_master_v4.csv
✅ Hierarchy Edge CSV found at: /content/drive/MyDrive/NICE/snomed_parent_child_edges_clean.csv
✅ Chroma DB found at: /content/drive/MyDrive/NICE/chroma_db_v4


---
## Section 3 · Pipeline Code

The cells below inline the pipeline logic from the project source files.  
Run them in order — each cell adds one layer of the stack.

In [135]:
# ── Policy Constants (revised, simplified) ───────────────────────────────────

# Query-context triggers
QUERY_HISTORY_TERMS = {
    'history of', 'history', 'past', 'previous', 'resolved', 'remission', 'follow-up'}
QUERY_CAUSAL_TERMS = (
    'associated with', 'due to','caused by', 'secondary to',)

QUERY_PREGNANCY_TERMS = ( 'pregnancy','childbirth','pre-existing', 'complicating pregnancy','gestational','maternal',)

# Retrieval-time exception handling
RETRIEVAL_HISTORY_EXCEPTION_TERMS = {
    'resolved', 'remission', 'history', 'past', 'follow', 'inactive'
}

RETRIEVAL_PREGNANCY_EXCEPTION_TERMS = {
    'pregnancy', 'childbirth', 'puerperium', 'eclampsia', 'pre', 'partum'
}
RETRIEVAL_PREGNANCY_MARKERS = ('pregnancy','childbirth','puerperium','eclampsia','pre-eclampsia','preeclampsia',
)

# Final output / presentation caps
DEFAULT_CANDIDATE_POOL_LIMIT   = 20
DEFAULT_INCLUDE_CANDIDATES_CAP = 6
DEFAULT_REVIEW_CANDIDATES_CAP  = 6
DEFAULT_SPECIFIC_VARIANTS_CAP  = 6

In [136]:
# ── Query Planning (revised) ─────────────────────────────────────────────────
# Key changes from original:
#
#   1. No combined jobs by default.
#      "obesity with hypertension" as a retrieval query causes BM25 and
#      semantic search to surface compound SNOMED concepts (e.g. "pulmonary
#      hypertension with extreme obesity") that token-match both conditions.
#      Each condition now gets a bare, isolated search job.
#
#   2. Per-condition query_profile.
#      The original built one global profile from all conditions combined.
#      "hypertension"'s retrieval job was therefore shaped by obesity tokens.
#      Now each job carries a profile derived only from its own condition text.
#
#   3. query_weight decoupled from retrieval.
#      Previously weighted_retrieval_score = adj_score * query_weight was used
#      to sort candidates within each retrieval batch. That meant secondary
#      condition candidates were pushed down before fusion even saw them.
#      Retrieval quality and fusion-time importance are separate concerns.
#      query_weight is now stored as metadata only; HybridRetriever sorts by
#      adjusted_retrieval_score (pure retrieval signal).

from __future__ import annotations

import json
import logging
import re
from typing import Any

logger = logging.getLogger(__name__)

STOPWORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'by', 'for', 'from',
    'in', 'of', 'on', 'or', 'the', 'to', 'with',
}


def normalize_query(raw_query: str) -> str:
    return ' '.join(str(raw_query).strip().split())


def tokenize_text(text: str) -> set[str]:
    return {
        token
        for token in re.findall(r'[a-z0-9]+', str(text).lower())
        if token and token not in STOPWORDS and len(token) > 1
    }


# ── QueryDecomposer ───────────────────────────────────────────────────────────
# Unchanged from original — the decomposition logic is sound.
# The problems were downstream in how the planner used its output.

class QueryDecomposer:
    def __init__(self, llm_model: str):
        self.llm_model = llm_model
        self._llm = None

    def _get_llm(self):
        if self._llm is False:
            return None
        if self._llm is None:
            try:
                from langchain_ollama import ChatOllama
                self._llm = ChatOllama(model=self.llm_model, temperature=0, format='json')
            except Exception as exc:
                logger.warning('Could not initialise decomposition LLM: %s', exc)
                self._llm = False
        return self._llm if self._llm is not False else None

    @staticmethod
    def _clean_list(values: Any) -> list[str]:
        if not isinstance(values, list):
            return []
        return list(dict.fromkeys(normalize_query(v) for v in values if normalize_query(v)))

    @staticmethod
    def _split_condition_candidates(text: str) -> list[str]:
        normalized = normalize_query(text)
        if not normalized:
            return []
        protected = normalized
        protected = re.sub(r'\bpoorly controlled\b', 'poorly_controlled', protected, flags=re.IGNORECASE)
        protected = re.sub(r'\btype 1\b', 'type_1', protected, flags=re.IGNORECASE)
        protected = re.sub(r'\btype 2\b', 'type_2', protected, flags=re.IGNORECASE)
        parts = re.split(r'\s*,\s*|\s+\band\b\s+', protected, flags=re.IGNORECASE)
        cleaned = []
        for part in parts:
            restored = (
                part.replace('poorly_controlled', 'poorly controlled')
                    .replace('type_1', 'type 1')
                    .replace('type_2', 'type 2')
            )
            restored = normalize_query(restored)
            if restored:
                cleaned.append(restored)
        return list(dict.fromkeys(cleaned))

    @classmethod
    def _extract_leading_modifiers(cls, segment: str) -> tuple[list[str], str]:
        tokens = normalize_query(segment).split()
        if not tokens:
            return [], ''
        known_modifiers = {
            'severe', 'mild', 'moderate', 'morbid', 'central',
            'poorly', 'controlled', 'uncontrolled',
        }
        modifier_tokens: list[str] = []
        while tokens and tokens[0].lower() in known_modifiers:
            modifier_tokens.append(tokens.pop(0))
        mods = cls._clean_list([' '.join(modifier_tokens)]) if modifier_tokens else []
        return mods, ' '.join(tokens)

    def _fallback_decompose(self, query: str) -> dict[str, Any]:
        normalized_query = normalize_query(query)
        modifiers: list[str] = []
        supporting_terms: list[str] = []
        segments = re.split(r'\bwith\b', normalized_query, maxsplit=1, flags=re.IGNORECASE)
        primary_segment  = normalize_query(segments[0]) if segments else normalized_query
        tail_segment     = normalize_query(segments[1]) if len(segments) > 1 else ''
        head_conditions  = self._split_condition_candidates(primary_segment)
        tail_conditions  = self._split_condition_candidates(tail_segment) if tail_segment else []
        primary_condition    = head_conditions[0] if head_conditions else primary_segment or normalized_query
        secondary_conditions = head_conditions[1:] + tail_conditions
        extracted_modifiers, stripped_primary = self._extract_leading_modifiers(primary_condition)
        if stripped_primary:
            primary_condition = stripped_primary
            modifiers.extend(extracted_modifiers)
        normalized_secondary: list[str] = []
        for secondary in secondary_conditions:
            ext_mods, stripped = self._extract_leading_modifiers(secondary)
            if stripped:
                normalized_secondary.append(stripped)
                supporting_terms.extend(ext_mods)
            elif secondary:
                normalized_secondary.append(secondary)
        return {
            'original_query': normalized_query,
            'primary_condition': primary_condition or normalized_query,
            'secondary_conditions': self._clean_list(normalized_secondary),
            'modifiers': self._clean_list(modifiers),
            'supporting_terms': self._clean_list(supporting_terms),
        }

    def _normalize_schema(self, parsed: dict[str, Any], fallback: dict[str, Any]) -> dict[str, Any]:
        primary = normalize_query(parsed.get('primary_condition', '')) or fallback['primary_condition']
        normalized = {
            'original_query': fallback['original_query'],
            'primary_condition': primary,
            'secondary_conditions': self._clean_list(parsed.get('secondary_conditions', [])),
            'modifiers': self._clean_list(parsed.get('modifiers', [])),
            'supporting_terms': self._clean_list(parsed.get('supporting_terms', [])),
        }
        if not normalized['secondary_conditions'] and parsed.get('comorbidities'):
            normalized['secondary_conditions'] = self._clean_list(parsed.get('comorbidities', []))
        if (
            normalized['primary_condition'].lower() == fallback['original_query'].lower()
            and not normalized['secondary_conditions']
            and fallback['secondary_conditions']
        ):
            return fallback
        return normalized

    def decompose(self, query: str) -> dict[str, Any]:
        fallback = self._fallback_decompose(query)
        llm = self._get_llm()
        if llm is None:
            return fallback
        schema_lines = [
            '{',
            '  "primary_condition": "string",',
            '  "secondary_conditions": ["string"],',
            '  "modifiers": ["string"],',
            '  "supporting_terms": ["string"]',
            '}',
        ]
        prompt = '\n'.join([
            'You are a clinical query decomposition assistant.',
            'Return ONLY valid JSON using this exact schema:',
            '\n'.join(schema_lines),
            'Guidance: no markdown, no commentary. Keep the schema lean.',
            'User query: "' + query + '"',
        ])
        try:
            response = llm.invoke(prompt)
            raw_text = response.content if hasattr(response, 'content') else str(response)
            parsed = json.loads(raw_text.strip().replace('```json', '').replace('```', ''))
            return self._normalize_schema(parsed, fallback)
        except Exception as exc:
            logger.warning('Decomposition failed; using fallback. %s', exc)
            return fallback


# ── QueryPlanner ──────────────────────────────────────────────────────────────

class QueryPlanner:
    def __init__(
        self,
        query_type_weights: dict[str, float],
        include_combined_jobs: bool = False,
    ):
        self.query_type_weights   = query_type_weights
        # Combined jobs (e.g. "obesity with hypertension") are OFF by default.
        # They cause BM25 and semantic search to surface compound SNOMED concepts
        # that happen to token-match both conditions but are clinically irrelevant
        # as broad anchors for either. Enable only if cross-condition enrichment
        # is explicitly needed and downstream scoring can handle the noise.
        self.include_combined_jobs = include_combined_jobs

    @staticmethod
    def _build_condition_profile(condition_text: str) -> dict[str, Any]:
        """
        Build a query profile scoped to a single condition's tokens only.

        Previously this was built from all conditions combined, meaning
        "hypertension"'s retrieval job carried obesity tokens in its profile.
        That distorted tag preferences and blocked/tolerated sets for every job.
        """
        tokens = tokenize_text(condition_text)

        infection_terms  = {'infection', 'infective', 'organism', 'bacteria', 'bacterial',
                            'virus', 'viral', 'fungal', 'microbiology'}
        medication_terms = {'drug', 'medication', 'medicine', 'tablet', 'capsule', 'insulin'}
        procedure_terms  = {'procedure', 'operation', 'surgery', 'therapy', 'treatment', 'screening'}

        preferred_tags = {'disorder', 'finding'}
        tolerated_tags = {'situation', 'observable entity'}
        blocked_tags   = {
            'organism', 'substance', 'qualifier value', 'occupation',
            'physical object', 'environment', 'specimen', 'attribute', 'assessment scale',
        }

        if tokens & infection_terms:
            preferred_tags.add('organism')
            blocked_tags.discard('organism')
        if tokens & medication_terms:
            tolerated_tags.update({'clinical drug', 'medicinal product', 'substance'})
            blocked_tags.difference_update({'clinical drug', 'medicinal product', 'substance'})
        if tokens & procedure_terms:
            tolerated_tags.update({'procedure', 'regime/therapy'})

        return {
            'tokens': sorted(tokens),
            'preferred_tags': sorted(preferred_tags),
            'tolerated_tags': sorted(tolerated_tags),
            'blocked_tags': sorted(blocked_tags),
        }

    def build_search_queries(self, structured_query: dict[str, Any]) -> list[dict[str, Any]]:
        primary              = normalize_query(structured_query.get('primary_condition', ''))
        secondary_conditions = structured_query.get('secondary_conditions', [])
        modifiers            = structured_query.get('modifiers', [])
        supporting_terms     = structured_query.get('supporting_terms', [])

        jobs: list[dict[str, Any]] = []
        seen: set[tuple[str, str]] = set()

        def add_job(
            query_text: str,
            query_type: str,
            clinical_focus: str,
            condition_profile: dict[str, Any],
        ) -> None:
            normalized_text = normalize_query(query_text)
            if not normalized_text:
                return
            key = (normalized_text.lower(), query_type)
            if key in seen:
                return
            seen.add(key)
            jobs.append({
                'query_text':      normalized_text,
                'query_type':      query_type,
                'clinical_focus':  clinical_focus,
                # query_weight is metadata for fusion; it must NOT affect retrieval
                # sort order. HybridRetriever sorts by adjusted_retrieval_score only.
                'query_weight':    self.query_type_weights.get(query_type, 1.0),
                'query_terms':     sorted(tokenize_text(normalized_text)),
                # Per-condition profile — scoped only to this condition's tokens.
                'query_profile':   condition_profile,
            })

        # ── Primary condition ─────────────────────────────────────────────────
        if primary:
            primary_profile = self._build_condition_profile(primary)
            add_job(primary, 'primary_condition', primary, primary_profile)

            # Modifier variants: "morbid obesity", "poorly controlled diabetes"
            # These stay attached to their own condition — not cross-condition.
            for modifier in modifiers:
                add_job(f'{modifier} {primary}', 'modifier', primary, primary_profile)

        # ── Secondary conditions — each fully isolated ────────────────────────
        # Each secondary condition gets its own bare query with its own profile.
        # This is the key fix: "hypertension" retrieves against SNOMED as a clean
        # single-condition query, without obesity tokens in the query text or profile.
        for secondary in secondary_conditions:
            secondary_profile = self._build_condition_profile(secondary)
            add_job(secondary, 'secondary_condition', secondary, secondary_profile)

            for modifier in modifiers:
                add_job(f'{modifier} {secondary}', 'modifier', secondary, secondary_profile)

        # ── Supporting terms ──────────────────────────────────────────────────
        if primary and supporting_terms:
            primary_profile = self._build_condition_profile(primary)
            for term in supporting_terms:
                add_job(term, 'supporting_term', term, primary_profile)

        # ── Combined cross-condition jobs (disabled by default) ───────────────
        # Only enable if you have specific evidence that combined queries improve
        # recall for your use case AND downstream scoring penalises compound hits.
        if self.include_combined_jobs and primary:
            primary_profile = self._build_condition_profile(primary)
            for secondary in secondary_conditions:
                add_job(
                    f'{primary} with {secondary}',
                    'combined',
                    secondary,
                    primary_profile,
                )

        return jobs

In [137]:
# ── Retrieval Engine (revised) ───────────────────────────────────────────────
# Key changes from original:
#
#   1. _lexical_overlap now checks the candidate TERM only, not text_for_embedding.
#      The enriched text field includes opencodelist_clinical_areas and
#      qof_cluster_description. A concept whose term is irrelevant but whose QOF
#      cluster description mentions the query condition gets undeserved overlap
#      credit. Overlap must measure term-to-query fit, not enriched field noise.
#
#   2. _term_precision added as a new retrieval signal.
#      _lexical_overlap is recall-oriented: how many query tokens appear in the
#      candidate? _term_precision is precision-oriented: what fraction of the
#      candidate's own tokens are query tokens? Together they penalise compound
#      concepts without any disease-specific rules.
#      Example for query "hypertension":
#        "Hypertension (disorder)"              overlap=1.0  precision=1.0
#        "Essential hypertension"               overlap=1.0  precision=0.5
#        "Pulmonary hypertension with extreme obesity" overlap=1.0  precision=0.2
#
#   3. _specificity_score uses num_parents and num_children from the dataframe.
#      These fields are already in snomed_master_v4. They distinguish a broad
#      anchor (few parents, many children) from a deep leaf (many parents, no
#      children) without any hardcoded disease lists.
#
#   4. Retrieval sort order decoupled from query_weight.
#      Previously candidates were sorted by weighted_retrieval_score = adj_score
#      * query_weight. Secondary condition jobs have lower query_weight, so
#      hypertension candidates were pushed down within their own batch before
#      fusion. Retrieval should return the best candidates by retrieval quality.
#      Fusion decides importance. Sort is now by adjusted_retrieval_score only;
#      query_weight is preserved as a field for downstream use.
#
#   5. Evidence bonus (+0.05) removed from adj_score.
#      It added a flat undirected bonus for any concept with any evidence source,
#      regardless of relevance. That logic belongs in authority scoring (decision
#      engine), not retrieval. Retrieval should measure fit to the query, not
#      source quality.

from __future__ import annotations

import logging
import math
from pathlib import Path
from typing import Any

import chromadb
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

logger = logging.getLogger(__name__)

STOPWORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'by', 'for', 'from',
    'in', 'of', 'on', 'or', 'the', 'to', 'with',
}


def normalize_query(raw_query: str) -> str:
    return ' '.join(str(raw_query).strip().split())


def tokenize_text(text: str) -> set[str]:
    return {
        token
        for token in __import__('re').findall(r'[a-z0-9]+', str(text).lower())
        if token and token not in STOPWORDS and len(token) > 1
    }


# ── DataLoader ────────────────────────────────────────────────────────────────
# Unchanged except: num_parents and num_children are now explicitly loaded and
# normalised so _specificity_score can use them in HybridRetriever.

class DataLoader:
    _shared_df: pd.DataFrame | None            = None
    _shared_df_by_code: pd.DataFrame | None    = None
    _shared_bm25: BM25Okapi | None             = None
    _shared_collection                         = None
    _shared_embedding_model: SentenceTransformer | None = None

    def __init__(self, config: Any):
        self.config = config

    @staticmethod
    def _safe_text(value: Any) -> str:
        return '' if pd.isna(value) else str(value).strip()

    @staticmethod
    def _column_or_default(df: pd.DataFrame, column: str, default: Any) -> pd.Series:
        if column in df.columns:
            return df[column]
        return pd.Series([default] * len(df), index=df.index)

    def get_dataframe(self) -> pd.DataFrame | None:
        if DataLoader._shared_df is not None:
            return DataLoader._shared_df

        try:
            df = pd.read_csv(self.config.snomed_path)
        except Exception as exc:
            logger.warning('Could not load SNOMED CSV from %s: %s', self.config.snomed_path, exc)
            return None

        if 'snomed_code' not in df.columns or 'term' not in df.columns:
            logger.warning('SNOMED CSV is missing required columns.')
            return None

        # ── Core cleanup ─────────────────────────────────────────────────────────
        df = df.dropna(subset=['snomed_code', 'term']).copy()

        df['snomed_code'] = df['snomed_code'].astype(str).str.strip()
        df['term']        = df['term'].astype(str).str.strip()

        df['semantic_tag']     = self._column_or_default(df, 'semantic_tag', '').fillna('').astype(str)
        df['in_qof']           = self._column_or_default(df, 'in_qof', False).fillna(False).astype(bool)
        df['in_opencodelists'] = self._column_or_default(df, 'in_opencodelists', False).fillna(False).astype(bool)

        df['log_usage_nhs'] = pd.to_numeric(
            self._column_or_default(df, 'log_usage_nhs', 0.0),
            errors='coerce'
        ).fillna(0.0)

        if 'usage_count_nhs' not in df.columns:
            df['usage_count_nhs'] = np.exp(df['log_usage_nhs']) - 1

        df['usage_count_nhs'] = pd.to_numeric(
            df['usage_count_nhs'],
            errors='coerce'
        ).fillna(0.0)

        # ── Keep metadata (NOT used in retrieval anymore) ────────────────────────
        df['opencodelist_clinical_areas'] = self._column_or_default(
            df, 'opencodelist_clinical_areas', ''
        ).fillna('').astype(str)

        df['qof_cluster_description'] = self._column_or_default(
            df, 'qof_cluster_description', ''
        ).fillna('').astype(str)

        # ── Hierarchy fields (used downstream ONLY) ──────────────────────────────
        df['num_parents'] = pd.to_numeric(
            self._column_or_default(df, 'num_parents', 0),
            errors='coerce'
        ).fillna(0).astype(int)

        df['num_children'] = pd.to_numeric(
            self._column_or_default(df, 'num_children', 0),
            errors='coerce'
        ).fillna(0).astype(int)

        # ── Deduplicate ─────────────────────────────────────────────────────────
        df = df.drop_duplicates(subset=['snomed_code']).copy()

        # ── CLEAN TEXT FIELDS (THIS IS THE FIX) ──────────────────────────────────

        # Used for semantic retrieval (clean, clinical only)
        df['text_for_embedding'] = (
            df['term'].apply(self._safe_text) + ' | ' +
            df['semantic_tag'].apply(self._safe_text)
        ).str.strip()

        # Used for BM25 (STRICT term-level only)
        df['text_for_bm25'] = (
            df['term'].apply(self._safe_text)
        ).str.strip()

        # ── Cache ───────────────────────────────────────────────────────────────
        DataLoader._shared_df = df
        DataLoader._shared_df_by_code = df.set_index('snomed_code', drop=False)

        logger.info('Loaded %s SNOMED records.', len(df))

        return DataLoader._shared_df

    def get_dataframe_by_code(self) -> pd.DataFrame | None:
        if DataLoader._shared_df_by_code is None:
            self.get_dataframe()
        return DataLoader._shared_df_by_code

    def get_bm25(self) -> BM25Okapi | None:
        if DataLoader._shared_bm25 is not None:
            return DataLoader._shared_bm25
        df = self.get_dataframe()
        if df is None:
            return None
        tokenized_corpus = [
            str(t).lower().split()
            for t in df['text_for_embedding'].fillna('').tolist()
        ]
        DataLoader._shared_bm25 = BM25Okapi(tokenized_corpus)
        logger.info('Initialised BM25 over %s records.', len(tokenized_corpus))
        return DataLoader._shared_bm25

    def get_collection(self):
        if DataLoader._shared_collection is not None:
            return DataLoader._shared_collection
        try:
            chroma_dir = Path(self.config.chroma_persist_dir)
            if chroma_dir.exists() and any(chroma_dir.iterdir()):
                logger.info('Loading existing Chroma DB from %s.', chroma_dir)
            else:
                logger.info('Chroma DB not found at %s — a fresh instance will be created.', chroma_dir)
            client = chromadb.PersistentClient(path=str(chroma_dir))
            DataLoader._shared_collection = client.get_or_create_collection(
                name=self.config.chroma_collection_name
            )
            return DataLoader._shared_collection
        except Exception as exc:
            logger.warning(
                'Could not load/create Chroma collection from %s: %s',
                self.config.chroma_persist_dir, exc,
            )
            return None

    def get_embedding_model(self) -> SentenceTransformer | None:
        if DataLoader._shared_embedding_model is not None:
            return DataLoader._shared_embedding_model
        try:
            embeddings_dir  = Path(self.config.embeddings_dir)
            model_cache_name = 'models--' + self.config.embedding_model_name.replace('/', '--')
            is_direct_model  = embeddings_dir.exists() and (
                (embeddings_dir / 'modules.json').exists() or
                (embeddings_dir / 'config.json').exists()
            )
            is_cached_model  = embeddings_dir.exists() and (
                embeddings_dir / model_cache_name
            ).exists()
            if is_direct_model:
                logger.info('Loading direct local embedding model from %s.', embeddings_dir)
                DataLoader._shared_embedding_model = SentenceTransformer(str(embeddings_dir))
            elif is_cached_model:
                logger.info('Found cached embedding model — reusing.')
                DataLoader._shared_embedding_model = SentenceTransformer(
                    self.config.embedding_model_name, cache_folder=str(embeddings_dir)
                )
            else:
                logger.info('Downloading/initialising embedding model %s...', self.config.embedding_model_name)
                DataLoader._shared_embedding_model = SentenceTransformer(
                    self.config.embedding_model_name, cache_folder=str(embeddings_dir)
                )
            return DataLoader._shared_embedding_model
        except Exception as exc:
            logger.warning('Could not load embedding model: %s', exc)
            return None



# ── HybridRetriever (clean, CE-first, no leakage) ─────────────────────────────

class HybridRetriever:
    def __init__(self, config: Any, data_loader: DataLoader):
        self.config = config
        self.data_loader = data_loader

    @staticmethod
    def _semantic_tag_weight(semantic_tag: str, query_profile: dict[str, Any]) -> float:
        tag = str(semantic_tag).strip().lower()
        preferred = set(query_profile.get('preferred_tags', []))
        tolerated = set(query_profile.get('tolerated_tags', []))
        blocked = set(query_profile.get('blocked_tags', []))

        if tag in blocked:
            return 0.0
        if tag in preferred:
            return 1.15
        if tag in tolerated:
            return 0.75

        if tag in {'body structure', 'procedure', 'event', 'morphologic abnormality'}:
            return 0.4

        return 0.55

    @staticmethod
    def _lexical_overlap(query_terms: set[str], candidate_term: str) -> float:
        if not query_terms:
            return 0.0
        term_tokens = tokenize_text(candidate_term)
        if not term_tokens:
            return 0.0
        return len(query_terms & term_tokens) / len(query_terms)

    @staticmethod
    def _term_precision(query_terms: set[str], candidate_term: str) -> float:
        if not query_terms:
            return 0.0
        term_tokens = tokenize_text(candidate_term)
        if not term_tokens:
            return 0.0
        return len(query_terms & term_tokens) / len(term_tokens)

    @staticmethod
    def _should_suppress(term: str, query_terms: set[str]) -> bool:
        lowered = term.lower()

        HISTORY_TERMS = {'resolved', 'remission', 'history', 'past', 'follow', 'inactive'}
        PREGNANCY_TERMS = {'pregnancy', 'childbirth', 'puerperium', 'eclampsia'}

        if 'resolved' in lowered and not (query_terms & HISTORY_TERMS):
            return True

        if any(t in lowered for t in PREGNANCY_TERMS) and not (query_terms & PREGNANCY_TERMS):
            return True

        return False

    def retrieve(self, job: dict[str, Any], top_k: int) -> list[dict[str, Any]]:
        df_by_code = self.data_loader.get_dataframe_by_code()
        df = self.data_loader.get_dataframe()
        bm25 = self.data_loader.get_bm25()
        collection = self.data_loader.get_collection()
        embedder = self.data_loader.get_embedding_model()

        if any(r is None for r in (df_by_code, df, bm25, collection, embedder)):
            return []

        query_text = job['query_text']
        query_type = job['query_type']
        query_weight = float(job['query_weight'])
        clinical_focus = job['clinical_focus']
        query_terms = set(job.get('query_terms', []))
        query_profile = job.get('query_profile', {})

        # ── Semantic retrieval ────────────────────────────────────────────────
        semantic_scores = {}
        try:
            qe = embedder.encode(query_text).tolist()
            sr = collection.query(query_embeddings=[qe], n_results=top_k, include=['distances'])

            for idx, code in enumerate(sr.get('ids', [[]])[0]):
                d = float(sr['distances'][0][idx])
                semantic_scores[str(code)] = 1.0 / (1.0 + d)
        except Exception:
            pass

        # ── BM25 retrieval (TERM ONLY) ────────────────────────────────────────
        bm25_scores = {}
        try:
            query_tokens = list(tokenize_text(query_text))
            raw_scores = bm25.get_scores(query_tokens)

            top_idx = np.argsort(raw_scores)[::-1][:top_k]
            max_score = float(raw_scores[top_idx].max()) if len(top_idx) else 0.0

            for row_idx in top_idx:
                code = str(df.iloc[row_idx]['snomed_code'])
                bm25_scores[code] = raw_scores[row_idx] / max_score if max_score > 0 else 0.0
        except Exception:
            pass

        # ── Candidate scoring (STRICT relevance only) ─────────────────────────
        candidates = []

        for snomed_code in set(semantic_scores) | set(bm25_scores):

            if snomed_code not in df_by_code.index:
                continue

            row = df_by_code.loc[snomed_code]

            term = str(row['term'])
            semantic_tag = str(row.get('semantic_tag', ''))

            if self._should_suppress(term, query_terms):
                continue

            tag_weight = self._semantic_tag_weight(semantic_tag, query_profile)
            if tag_weight == 0.0:
                continue

            sem_score = semantic_scores.get(snomed_code, 0.0)
            bm25_score = bm25_scores.get(snomed_code, 0.0)

            # ── Base retrieval signal (NO additive mixing)
            base_retrieval = max(sem_score, bm25_score)

            lex_overlap = self._lexical_overlap(query_terms, term)
            term_prec = self._term_precision(query_terms, term)

            # ── HARD relevance gate
            if base_retrieval < 0.15 and lex_overlap == 0.0:
                continue

            # ── Multiplicative scoring (strict filtering)
            adj_score = (
                base_retrieval *
                tag_weight *
                (0.5 + 0.5 * term_prec) *
                (0.5 + 0.5 * lex_overlap)
            )

            if adj_score <= 0.0:
                continue

            candidates.append({
                'snomed_code': snomed_code,
                'term': term,
                'semantic_tag': semantic_tag,
                'in_qof': bool(row['in_qof']),
                'in_opencodelists': bool(row['in_opencodelists']),
                'usage_count_nhs': float(row['usage_count_nhs']),

                'query_text': query_text,
                'query_type': query_type,
                'clinical_focus': clinical_focus,
                'query_weight': query_weight,

                'semantic_score': float(sem_score),
                'bm25_score': float(bm25_score),
                'lexical_overlap': float(lex_overlap),
                'term_precision': float(term_prec),
                'semantic_tag_weight': float(tag_weight),

                'retrieval_score': float(base_retrieval),
                'adjusted_retrieval_score': float(adj_score),
                'weighted_retrieval_score': float(adj_score * query_weight),
            })

        # ── Sort by PURE retrieval quality ────────────────────────────────────
        candidates.sort(
            key=lambda c: (
                c['adjusted_retrieval_score'],
                c['term_precision'],
                c['lexical_overlap'],
                c['semantic_score'],
            ),
            reverse=True,
        )

        return candidates[:top_k]

In [138]:
# ── Hierarchy Enricher (revised) ──────────────────────────────────────────────
# Purpose:
# - add a small number of structural candidates
# - never inherit parent scores
# - use the bare clinical_focus for filtering, not the full query text

from __future__ import annotations

import math
from pathlib import Path
from typing import Any

import pandas as pd


class HierarchyEnricher:
    def __init__(self, snomed_path: str | Path, edge_path: str | Path):
        self.snomed_path = Path(snomed_path)
        self.edge_path = Path(edge_path)
        self._master = None
        self._master_by_code = None
        self._edge = None

    def _load(self) -> None:
        if self._master is None:
            master = pd.read_csv(self.snomed_path, dtype=str)
            master["snomed_code"] = master["snomed_code"].astype(str).str.strip()
            master["term"] = master["term"].astype(str).str.strip()
            master["semantic_tag"] = master.get("semantic_tag", "").fillna("").astype(str)

            for col in ["num_parents", "num_children"]:
                if col in master.columns:
                    master[col] = pd.to_numeric(master[col], errors="coerce").fillna(0).astype(int)
                else:
                    master[col] = 0

            for col in ["in_qof", "in_opencodelists"]:
                if col in master.columns:
                    master[col] = master[col].map(
                        {"True": True, "False": False, True: True, False: False}
                    ).fillna(False)
                else:
                    master[col] = False

            if "usage_count_nhs" in master.columns:
                master["usage_count_nhs"] = pd.to_numeric(
                    master["usage_count_nhs"], errors="coerce"
                ).fillna(0.0)
            else:
                master["usage_count_nhs"] = 0.0

            self._master = master
            self._master_by_code = master.set_index("snomed_code", drop=False)

        if self._edge is None:
            edge = pd.read_csv(self.edge_path, dtype=str)
            edge["child_code"] = edge["child_code"].astype(str).str.strip()
            edge["parent_code"] = edge["parent_code"].astype(str).str.strip()
            edge["child_term"] = edge["child_term"].astype(str).str.strip()
            edge["parent_term"] = edge["parent_term"].astype(str).str.strip()
            self._edge = edge

    @staticmethod
    def _normalize(text: str) -> str:
        return " ".join(str(text).strip().split())

    @staticmethod
    def _tokenize(text: str) -> set[str]:
        import re
        stop = {
            "a", "an", "and", "are", "as", "at", "by", "for", "from",
            "in", "of", "on", "or", "the", "to", "with",
        }
        return {
            t for t in re.findall(r"[a-z0-9]+", str(text).lower())
            if t and t not in stop and len(t) > 1
        }

    @classmethod
    def _lexical_overlap(cls, query_text: str, candidate_term: str) -> float:
        q = cls._tokenize(query_text)
        c = cls._tokenize(candidate_term)
        if not q or not c:
            return 0.0
        return len(q & c) / max(1, len(q))

    @classmethod
    def _term_precision(cls, query_text: str, candidate_term: str) -> float:
        q = cls._tokenize(query_text)
        c = cls._tokenize(candidate_term)
        if not q or not c:
            return 0.0
        return len(q & c) / max(1, len(c))

    @classmethod
    def _specificity_score(
        cls,
        candidate_term: str,
        condition_str: str,
        num_parents: int,
        num_children: int,
    ) -> float:
        stop_tokens = {"disorder", "finding", "observable", "entity", "situation"}
        extra_tokens = cls._tokenize(candidate_term) - cls._tokenize(condition_str) - stop_tokens
        depth_penalty = min(1.0, num_parents / 8.0)
        extra_penalty = min(1.0, len(extra_tokens) / 5.0)
        breadth_bonus = min(0.3, math.log1p(num_children) / 10.0)
        score = 1.0 - 0.45 * depth_penalty - 0.55 * extra_penalty + breadth_bonus
        return max(0.0, min(1.0, score))

    def get_concept_info(self, code: str) -> dict[str, Any] | None:
        self._load()
        code = str(code).strip()
        if code not in self._master_by_code.index:
            return None
        row = self._master_by_code.loc[code]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        return row.to_dict()

    def get_children(self, parent_code: str) -> list[dict[str, Any]]:
        self._load()
        children = self._edge.loc[
            self._edge["parent_code"] == str(parent_code).strip(),
            ["child_code", "child_term", "parent_code", "parent_term"],
        ].drop_duplicates()
        return children.to_dict(orient="records")

    def enrich_batch(
        self,
        batch: list[dict[str, Any]],
        top_n_parents_per_focus: int = 1,
        max_children_per_parent: int = 3,
        min_child_overlap: float = 0.10,
    ) -> list[dict[str, Any]]:
        """
        Adds child candidates as zero-trust recall candidates.
        They do not inherit parent score.
        They must pass a small lexical check against clinical_focus.
        """
        self._load()

        if not batch:
            return batch

        enriched = list(batch)
        seen_codes = {str(c["snomed_code"]).strip() for c in batch}

        # group by clinical_focus so multimorbidity does not get hijacked by one condition
        by_focus: dict[str, list[dict[str, Any]]] = {}
        for c in batch:
            focus = self._normalize(c.get("clinical_focus", ""))
            by_focus.setdefault(focus, []).append(c)

        for focus, focus_batch in by_focus.items():
            # direct retrieved candidates only
            direct_candidates = [
                c for c in focus_batch
                if c.get("retrieval_method") != "hierarchy_child"
            ]

            ranked_parents = sorted(
                direct_candidates,
                key=lambda x: float(x.get("adjusted_retrieval_score", 0.0)),
                reverse=True,
            )

            parents_taken = 0
            for parent_candidate in ranked_parents:
                if parents_taken >= top_n_parents_per_focus:
                    break

                parent_code = str(parent_candidate["snomed_code"]).strip()
                parent_info = self.get_concept_info(parent_code)
                if not parent_info:
                    continue

                if int(parent_info.get("num_children", 0) or 0) <= 0:
                    continue

                parents_taken += 1
                children = self.get_children(parent_code)[:max_children_per_parent]

                for child in children:
                    child_code = str(child["child_code"]).strip()
                    if child_code in seen_codes:
                        continue

                    child_info = self.get_concept_info(child_code)
                    if not child_info:
                        continue

                    child_term = str(child_info.get("term", child["child_term"]))
                    lex = self._lexical_overlap(focus, child_term)
                    if lex < min_child_overlap:
                        continue

                    prec = self._term_precision(focus, child_term)
                    spec = self._specificity_score(
                        child_term,
                        focus,
                        int(child_info.get("num_parents", 0) or 0),
                        int(child_info.get("num_children", 0) or 0),
                    )

                    enriched.append({
                        "snomed_code": child_code,
                        "term": child_term,
                        "semantic_tag": str(child_info.get("semantic_tag", "")),
                        "in_qof": bool(child_info.get("in_qof", False)),
                        "in_opencodelists": bool(child_info.get("in_opencodelists", False)),
                        "usage_count_nhs": float(child_info.get("usage_count_nhs", 0.0) or 0.0),
                        "num_parents": int(child_info.get("num_parents", 0) or 0),
                        "num_children": int(child_info.get("num_children", 0) or 0),
                        "query_text": parent_candidate["query_text"],
                        "query_type": parent_candidate["query_type"],
                        "clinical_focus": focus,
                        "query_weight": parent_candidate.get("query_weight", 1.0),

                        # zero inherited trust
                        "semantic_score": 0.0,
                        "bm25_score": 0.0,
                        "retrieval_score": 0.0,
                        "adjusted_retrieval_score": 0.0,
                        "weighted_retrieval_score": 0.0,

                        # useful metadata only
                        "lexical_overlap": float(lex),
                        "term_precision": float(prec),
                        "specificity_score": float(spec),
                        "semantic_tag_weight": 0.0,

                        "retrieval_method": "hierarchy_child",
                        "hierarchy_parent_code": parent_code,
                        "hierarchy_parent_term": parent_candidate["term"],
                    })
                    seen_codes.add(child_code)

        return enriched

In [139]:
# ── Candidate Fusion (revised) ────────────────────────────────────────────────
# Purpose:
# - merge recall candidates
# - deduplicate by SNOMED code
# - preserve provenance
# - use rank-based fusion, not raw score blending

from __future__ import annotations

from collections import defaultdict
from typing import Any


class CandidateFusionEngine:
    def __init__(self, rrf_k: int = 60):
        self.rrf_k = rrf_k

    def fuse(self, retrieval_batches: list[list[dict[str, Any]]]) -> list[dict[str, Any]]:
        fused: dict[str, dict[str, Any]] = {}

        for batch_index, batch in enumerate(retrieval_batches):
            if not batch:
                continue

            for rank, c in enumerate(batch, start=1):
                code = str(c["snomed_code"]).strip()
                item = fused.setdefault(code, {
                    "snomed_code": code,
                    "term": c["term"],
                    "semantic_tag": c.get("semantic_tag", ""),
                    "in_qof": c.get("in_qof", False),
                    "in_opencodelists": c.get("in_opencodelists", False),
                    "usage_count_nhs": c.get("usage_count_nhs", 0.0),
                    "num_parents": c.get("num_parents", 0),
                    "num_children": c.get("num_children", 0),

                    "query_types_hit": set(),
                    "query_texts_hit": set(),
                    "clinical_focuses_hit": set(),
                    "retrieval_trace": [],

                    "semantic_score_max": 0.0,
                    "bm25_score_max": 0.0,
                    "lexical_overlap_max": 0.0,
                    "term_precision_max": 0.0,
                    "specificity_score_max": 0.0,
                    "best_adjusted_retrieval_score": 0.0,

                    "rrf_score": 0.0,
                    "has_direct_retrieval": False,
                    "has_hierarchy_retrieval": False,
                    "hierarchy_parent_code": None,
                    "hierarchy_parent_term": None,
                })

                item["query_types_hit"].add(c.get("query_type"))
                item["query_texts_hit"].add(c.get("query_text"))
                item["clinical_focuses_hit"].add(c.get("clinical_focus"))

                item["semantic_score_max"] = max(item["semantic_score_max"], float(c.get("semantic_score", 0.0)))
                item["bm25_score_max"] = max(item["bm25_score_max"], float(c.get("bm25_score", 0.0)))
                item["lexical_overlap_max"] = max(item["lexical_overlap_max"], float(c.get("lexical_overlap", 0.0)))
                item["term_precision_max"] = max(item["term_precision_max"], float(c.get("term_precision", 0.0)))
                item["specificity_score_max"] = max(item["specificity_score_max"], float(c.get("specificity_score", 0.0)))
                item["best_adjusted_retrieval_score"] = max(
                    item["best_adjusted_retrieval_score"],
                    float(c.get("adjusted_retrieval_score", 0.0)),
                )

                retrieval_method = c.get("retrieval_method", "direct")
                if retrieval_method == "hierarchy_child":
                    item["has_hierarchy_retrieval"] = True
                    item["hierarchy_parent_code"] = c.get("hierarchy_parent_code")
                    item["hierarchy_parent_term"] = c.get("hierarchy_parent_term")
                else:
                    item["has_direct_retrieval"] = True

                # rank-based fusion
                item["rrf_score"] += 1.0 / (self.rrf_k + rank)

                item["retrieval_trace"].append({
                    "batch_index": batch_index,
                    "rank_in_batch": rank,
                    "query_text": c.get("query_text"),
                    "query_type": c.get("query_type"),
                    "clinical_focus": c.get("clinical_focus"),
                    "retrieval_method": retrieval_method,
                    "semantic_score": round(float(c.get("semantic_score", 0.0)), 6),
                    "bm25_score": round(float(c.get("bm25_score", 0.0)), 6),
                    "lexical_overlap": round(float(c.get("lexical_overlap", 0.0)), 6),
                    "term_precision": round(float(c.get("term_precision", 0.0)), 6),
                    "specificity_score": round(float(c.get("specificity_score", 0.0)), 6),
                    "adjusted_retrieval_score": round(float(c.get("adjusted_retrieval_score", 0.0)), 6),
                })

        results: list[dict[str, Any]] = []
        for item in fused.values():
            results.append({
                "snomed_code": item["snomed_code"],
                "term": item["term"],
                "semantic_tag": item["semantic_tag"],
                "in_qof": item["in_qof"],
                "in_opencodelists": item["in_opencodelists"],
                "usage_count_nhs": item["usage_count_nhs"],
                "num_parents": item["num_parents"],
                "num_children": item["num_children"],

                # fusion output
                "fusion_score": round(float(item["rrf_score"]), 6),
                "rrf_score": round(float(item["rrf_score"]), 6),

                # carry max per-signal values for later decision/rerank use
                "semantic_score": round(float(item["semantic_score_max"]), 6),
                "bm25_score": round(float(item["bm25_score_max"]), 6),
                "lexical_overlap": round(float(item["lexical_overlap_max"]), 6),
                "term_precision": round(float(item["term_precision_max"]), 6),
                "specificity_score": round(float(item["specificity_score_max"]), 6),

                "semantic_score_max": round(float(item["semantic_score_max"]), 6),
                "bm25_score_max": round(float(item["bm25_score_max"]), 6),
                "lexical_overlap_max": round(float(item["lexical_overlap_max"]), 6),
                "term_precision_max": round(float(item["term_precision_max"]), 6),
                "specificity_score_max": round(float(item["specificity_score_max"]), 6),
                "best_adjusted_retrieval_score": round(float(item["best_adjusted_retrieval_score"]), 6),

                "query_coverage_count": len(item["clinical_focuses_hit"]),
                "query_types_hit": sorted(x for x in item["query_types_hit"] if x is not None),
                "clinical_focuses_hit": sorted(x for x in item["clinical_focuses_hit"] if x is not None),

                "has_direct_retrieval": item["has_direct_retrieval"],
                "has_hierarchy_retrieval": item["has_hierarchy_retrieval"],
                "retrieval_method": (
                    "direct+hierarchy"
                    if item["has_direct_retrieval"] and item["has_hierarchy_retrieval"]
                    else "hierarchy_child"
                    if item["has_hierarchy_retrieval"]
                    else "direct"
                ),
                "hierarchy_parent_code": item["hierarchy_parent_code"],
                "hierarchy_parent_term": item["hierarchy_parent_term"],
                "retrieval_trace": item["retrieval_trace"],
            })

        results.sort(
            key=lambda x: (
                x["fusion_score"],
                x["best_adjusted_retrieval_score"],
                x["term_precision_max"],
                x["lexical_overlap_max"],
                x["semantic_score_max"],
            ),
            reverse=True,
        )
        return results

In [140]:
# ── Relevance Gate / Reranker (plug-in version) ──────────────────────────────
# Returns a flat reranked shortlist, but builds it per condition first.

from __future__ import annotations
from typing import Any


class RelevanceGateReranker:
    def __init__(
        self,
        relevance_threshold: float = 0.25,
        top_n_per_condition: int = 50,
    ):
        self.relevance_threshold = relevance_threshold
        self.top_n_per_condition = top_n_per_condition

    @staticmethod
    def _norm(text: str) -> str:
        return normalize_query(text).lower()

    def _per_condition_scores(
        self,
        candidate: dict[str, Any],
        condition: str,
    ) -> dict[str, float]:
        """
        Score one candidate against one bare condition only.
        Relevance stays primary. Specificity only scales it modestly.
        """
        condition_n = self._norm(condition)

        semantic = 0.0
        bm25 = 0.0
        term_precision = 0.0
        specificity = float(candidate.get("specificity_score_max", 0.0) or 0.0)

        for trace in candidate.get("retrieval_trace", []):
            focus = self._norm(trace.get("clinical_focus", ""))
            if focus != condition_n:
                continue

            semantic = max(semantic, float(trace.get("semantic_score", 0.0) or 0.0))
            bm25 = max(bm25, float(trace.get("bm25_score", 0.0) or 0.0))
            term_precision = max(term_precision, float(trace.get("term_precision", 0.0) or 0.0))
            specificity = max(specificity, float(trace.get("specificity_score", 0.0) or 0.0))

        # Base relevance: semantic + BM25 + term precision
        relevance = (
            0.50 * semantic +
            0.30 * bm25 +
            0.20 * term_precision
        )

        # Specificity is a bounded modifier, not a free bonus
        rerank_score = relevance * (0.60 + 0.40 * specificity)

        return {
            "semantic": round(semantic, 6),
            "bm25": round(bm25, 6),
            "term_precision": round(term_precision, 6),
            "specificity_score": round(specificity, 6),
            "relevance_score": round(relevance, 6),
            "rerank_score": round(rerank_score, 6),
        }

    def rerank(
        self,
        fused_candidates: list[dict[str, Any]],
        structured_query: dict[str, Any],
    ) -> list[dict[str, Any]]:
        """
        Returns a flat reranked shortlist for compatibility with the current
        DecisionEngine, but builds that shortlist per condition first.
        """
        conditions = major_conditions(structured_query)
        if not conditions:
            return []

        scored: list[dict[str, Any]] = []

        # Score every candidate against every condition
        for candidate in fused_candidates:
            per_condition = {
                condition: self._per_condition_scores(candidate, condition)
                for condition in conditions
            }

            matched_conditions = [
                condition
                for condition, scores in per_condition.items()
                if scores["relevance_score"] >= self.relevance_threshold
            ]

            if not matched_conditions:
                continue

            dominant_condition = max(
                matched_conditions,
                key=lambda c: per_condition[c]["rerank_score"]
            )

            d = dict(candidate)
            d["condition_relevance"] = per_condition
            d["matched_conditions_from_gate"] = matched_conditions
            d["dominant_condition_from_gate"] = dominant_condition
            d["relevance_score"] = per_condition[dominant_condition]["relevance_score"]
            d["rerank_score"] = per_condition[dominant_condition]["rerank_score"]
            scored.append(d)

        # Build shortlist per condition first
        shortlisted_by_condition: dict[str, list[dict[str, Any]]] = {}
        for condition in conditions:
            branch = [
                c for c in scored
                if condition in c["matched_conditions_from_gate"]
            ]
            branch.sort(
                key=lambda x: (
                    x["condition_relevance"][condition]["rerank_score"],
                    x["condition_relevance"][condition]["relevance_score"],
                    x.get("fusion_score", 0.0),
                ),
                reverse=True,
            )
            shortlisted_by_condition[condition] = branch[:self.top_n_per_condition]

        # Flatten after per-condition ranking
        # Keep best version of each code by rerank score
        best_by_code: dict[str, dict[str, Any]] = {}
        for condition in conditions:
            for candidate in shortlisted_by_condition[condition]:
                code = str(candidate["snomed_code"])
                if (
                    code not in best_by_code
                    or candidate["rerank_score"] > best_by_code[code]["rerank_score"]
                ):
                    best_by_code[code] = candidate

        final_candidates = list(best_by_code.values())
        final_candidates.sort(
            key=lambda x: (
                x["rerank_score"],
                x["relevance_score"],
                x.get("fusion_score", 0.0),
                x.get("specificity_score_max", 0.0),
            ),
            reverse=True,
        )

        return final_candidates

In [141]:
# ── CrossEncoderReranker (standalone plug-in) ────────────────────────────────
# Insert this AFTER your current RelevanceGateReranker.

from __future__ import annotations
from typing import Any
import torch
from sentence_transformers import CrossEncoder


class CrossEncoderReranker:
    def __init__(
        self,
        model_name: str = "BAAI/bge-reranker-v2-m3",
        top_n_per_condition: int = 20,
        batch_size: int = 16,
        max_length: int = 512,
        device: str | None = None,
    ):
        self.model_name = model_name
        self.top_n_per_condition = top_n_per_condition
        self.batch_size = batch_size
        self.max_length = max_length
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self._model: CrossEncoder | None = None

    def _get_model(self) -> CrossEncoder:
        if self._model is None:
            self._model = CrossEncoder(
                self.model_name,
                max_length=self.max_length,
                device=self.device,
                trust_remote_code=True,
            )
        return self._model

    @staticmethod
    def _norm(text: str) -> str:
        return normalize_query(text).lower()

    @staticmethod
    def _candidate_text(candidate: dict[str, Any]) -> str:
        """
        Minimal, clean clinical text for CE.
        No evidence, no usage, no admin noise.
        """
        term = str(candidate.get("term", "")).strip()
        semantic_tag = str(candidate.get("semantic_tag", "")).strip()
        return f"{term} ({semantic_tag})".strip()

    def _score_pairs(self, pairs: list[tuple[str, str]]) -> list[float]:
        if not pairs:
            return []
        model = self._get_model()
        scores = model.predict(
            pairs,
            batch_size=self.batch_size,
            show_progress_bar=False,
        )
        return [float(s) for s in scores]

    def rerank(
        self,
        gate_candidates: list[dict[str, Any]],
        structured_query: dict[str, Any],
    ) -> list[dict[str, Any]]:
        """
        Input:
            output of RelevanceGateReranker
        Output:
            reranked candidates using CE per condition ONLY
        """
        conditions = major_conditions(structured_query)

        if not conditions or not gate_candidates:
            return gate_candidates

        # ── Build per-condition candidate pools ──────────────────────────────
        candidates_by_condition: dict[str, list[dict[str, Any]]] = {}

        for condition in conditions:
            branch = [
                c for c in gate_candidates
                if condition in c.get("matched_conditions_from_gate", [])
            ]

            branch.sort(
                key=lambda x: (
                    x.get("condition_relevance", {}).get(condition, {}).get("rerank_score", 0.0),
                    x.get("condition_relevance", {}).get(condition, {}).get("relevance_score", 0.0),
                    x.get("fusion_score", 0.0),
                ),
                reverse=True,
            )

            candidates_by_condition[condition] = branch[:self.top_n_per_condition]

        # ── Cross-encoder scoring (STRICTLY per condition) ───────────────────
        scored_by_code: dict[str, dict[str, Any]] = {}

        for condition in conditions:
            branch = candidates_by_condition.get(condition, [])
            if not branch:
                continue

            candidate_texts = [self._candidate_text(c) for c in branch]
            condition_pairs = [(condition, text) for text in candidate_texts]
            condition_scores = self._score_pairs(condition_pairs)
            for candidate, ce_condition_score in zip(branch, condition_scores):
                #  HARD CE FILTER
                if ce_condition_score < 0.2:
                    continue
                code = str(candidate["snomed_code"])
                enriched = dict(candidate)

                ce_score_by_condition = dict(enriched.get("ce_score_by_condition", {}))
                ce_score_by_condition[condition] = {
                    "ce_condition_score": round(ce_condition_score, 6),
                    "ce_score": round(ce_condition_score, 6),
                }

                enriched["ce_score_by_condition"] = ce_score_by_condition

                # Keep best CE view per code
                prev = scored_by_code.get(code)

                if prev is None:
                    scored_by_code[code] = enriched
                else:
                    prev_best = float(prev.get("ce_score", float("-inf")))

                    if ce_condition_score > prev_best:
                        merged = dict(enriched)
                        merged_scores = dict(prev.get("ce_score_by_condition", {}))
                        merged_scores.update(ce_score_by_condition)
                        merged["ce_score_by_condition"] = merged_scores
                        scored_by_code[code] = merged
                    else:
                        prev_scores = dict(prev.get("ce_score_by_condition", {}))
                        prev_scores.update(ce_score_by_condition)
                        prev["ce_score_by_condition"] = prev_scores

        # ── Final aggregation ────────────────────────────────────────────────
        final_candidates: list[dict[str, Any]] = []

        for code, candidate in scored_by_code.items():
            ce_score_by_condition = candidate.get("ce_score_by_condition", {})

            matched_conditions_from_ce = [
                condition for condition, scores in ce_score_by_condition.items()
                if float(scores.get("ce_score", 0.0)) >= 0.2
            ]

            if not matched_conditions_from_ce:
                matched_conditions_from_ce = list(candidate.get("matched_conditions_from_gate", []))

            if matched_conditions_from_ce:
                dominant_condition_from_ce = max(
                    matched_conditions_from_ce,
                    key=lambda c: float(ce_score_by_condition.get(c, {}).get("ce_score", float("-inf")))
                )
            else:
                dominant_condition_from_ce = candidate.get("dominant_condition_from_gate", "unassigned")

            best_ce = max(
                [float(v.get("ce_score", float("-inf"))) for v in ce_score_by_condition.values()],
                default=0.0,
            )

            enriched = dict(candidate)
            enriched["ce_score"] = round(best_ce, 6)
            enriched["matched_conditions_from_ce"] = matched_conditions_from_ce
            enriched["dominant_condition_from_ce"] = dominant_condition_from_ce

            final_candidates.append(enriched)

        # ── Final sort ───────────────────────────────────────────────────────
        final_candidates.sort(
            key=lambda x: (
                x.get("ce_score", 0.0),
                x.get("rerank_score", 0.0),
                x.get("relevance_score", 0.0),
                x.get("fusion_score", 0.0),
                x.get("specificity_score_max", 0.0),
            ),
            reverse=True,
        )

        return final_candidates

In [142]:
# ── Decision Engine (final: CE + topology + policy aligned) ───────────────────

BLOCKED_CORE_TAGS = {
    "situation",
    "context-dependent category",
    "finding"
}


def assign_confidence(candidate: dict[str, Any]) -> str:
    in_qof = bool(candidate.get('in_qof', False))
    in_oc = bool(candidate.get('in_opencodelists', False))
    usage = float(candidate.get('usage_count_nhs', 0.0))

    if in_qof:
        return 'HIGH'
    if in_oc and usage >= 29097:
        return 'HIGH'
    if in_oc or usage >= 6730:
        return 'MEDIUM'
    return 'REVIEW'


def compute_bounded_authority(candidate: dict[str, Any]) -> float:
    usage = float(candidate.get('usage_count_nhs', 0.0))
    in_qof = bool(candidate.get('in_qof', False))
    in_oc = bool(candidate.get('in_opencodelists', False))

    return min(
        1.0,
        0.40 * float(in_qof) +
        0.25 * float(in_oc) +
        min(0.35, math.log10(usage + 1.0) / 8.0)
    )


class DecisionEngine:
    def assign_final_decisions(
        self,
        reranked_candidates: list[dict[str, Any]],
        structured_query: dict[str, Any],
        top_k: int,
    ) -> dict[str, list[dict[str, Any]]]:

        conditions = major_conditions(structured_query)
        multimorbidity = len(conditions) > 1

        # ── Stage 1: scoring (KEEP YOUR LOGIC) ───────────────────────────────
        enriched = []

        for candidate in reranked_candidates:
            ce_score = float(candidate.get('ce_score', 0.0))
            rerank_score = float(candidate.get('rerank_score', 0.0))
            relevance_score = float(candidate.get('relevance_score', 0.0))

            final_rerank_score = (
                0.70 * ce_score + 0.30 * rerank_score
            ) if ce_score > 0 else rerank_score
            # 🔴 HARD CE QUALITY FILTER
            if ce_score < 0.2:
                continue

            if final_rerank_score <= 0:
                continue

            matched = list(candidate.get('matched_conditions_from_ce', [])) or list(
                candidate.get('matched_conditions_from_gate', [])
            )

            if not matched:
                continue

            dominant = (
                candidate.get('dominant_condition_from_ce')
                or candidate.get('dominant_condition_from_gate')
                or 'unassigned'
            )

            authority = compute_bounded_authority(candidate)

            # --- scoring components ---
            coverage_bonus = (
                min(0.20, 0.08 * len(matched))
                if multimorbidity else
                min(0.10, 0.05 * len(matched))
            )

            specificity = float(candidate.get('specificity_score_max', 0.0))
            anchor_bonus = (
                0.10 if specificity >= 0.70 else
                0.05 if specificity >= 0.50 else
                0.0
            )

            presentation_score = (
                0.65 * final_rerank_score +
                0.20 * authority +
                0.10 * coverage_bonus +
                0.05 * anchor_bonus
            )

            ranking_components = {
                'ce_score': round(ce_score, 6),
                'rerank_score': round(rerank_score, 6),
                'final_rerank_score': round(final_rerank_score, 6),
                'relevance_score': round(relevance_score, 6),
                'authority_component': round(authority, 6),
                'coverage_component': round(coverage_bonus, 6),
                'anchor_bonus': round(anchor_bonus, 6),
            }

            d = dict(candidate)
            d['confidence_tier'] = assign_confidence(candidate)
            d['matched_conditions'] = matched
            d['dominant_condition'] = dominant
            d['final_rerank_score'] = final_rerank_score
            d['authority_score'] = authority
            d['presentation_score'] = round(presentation_score, 6)
            d['ranking_components'] = ranking_components

            enriched.append(d)

        # ── Stage 2: group per condition (CRITICAL FIX) ──────────────────────
        by_condition: dict[str, list[dict[str, Any]]] = {c: [] for c in conditions}

        for c in enriched:
            for cond in c['matched_conditions']:
                if cond in by_condition:
                    by_condition[cond].append(c)

        include, review, specific = [], [], []
        selected_codes = set()

        # ── Stage 3: strict anchor selection (1 per condition) ───────────────
        for condition in conditions:
            candidates = by_condition.get(condition, [])
            if not candidates:
                continue

            candidates = sorted(
                candidates,


                key=lambda x: (
                -x['ranking_components']['ce_score'],
                -x['authority_score']
            )
            )


            anchor = None

            for c in candidates:
                tag = str(c.get("semantic_tag", "")).lower()

                if tag not in BLOCKED_CORE_TAGS and c['final_rerank_score'] >= 0.2:
                    anchor = c
                    break

            if anchor:
                anchor['candidate_role'] = 'core'
                include.append(anchor)
                selected_codes.add(anchor['snomed_code'])

            # rest → variant / specific
            for c in candidates:
                if c['snomed_code'] in selected_codes:
                    continue

                tag = str(c.get("semantic_tag", "")).lower()

                if tag in BLOCKED_CORE_TAGS:
                    c['candidate_role'] = 'specific'
                    specific.append(c)
                else:
                    c['candidate_role'] = 'variant'
                    review.append(c)

        # ── Stage 4: deduplicate + cap ───────────────────────────────────────
        def serialize_list(rows, cap):
            out = []
            seen = set()

            for i, r in enumerate(rows, start=1):
                code = r['snomed_code']
                if code in seen:
                    continue

                seen.add(code)
                out.append(serialize_candidate_output(r, index=i))

                if len(out) >= cap:
                    break

            return out

        return {
            'include_candidates': serialize_list(include, len(conditions)),
            'review_candidates': serialize_list(review, 5),
            'specific_variants': serialize_list(specific, 5),
            'suppressed_candidates': [],
        }

In [143]:
# ── Scoring Rules (revised, aligned) ─────────────────────────────────────────
# Role now:
# - lightweight support utilities
# - query/context helpers
# - suppression helpers
# - output serialization for the revised pipeline


def major_conditions(structured_query: dict[str, Any]) -> list[str]:
    raw = [structured_query.get('primary_condition', '')] + structured_query.get('secondary_conditions', [])
    seen: set[str] = set()
    result = []
    for c in raw:
        t = normalize_query(c)
        if t and t.lower() not in seen:
            seen.add(t.lower())
            result.append(t)
    return result


def query_context(structured_query: dict[str, Any]) -> dict[str, Any]:
    raw_query = str(structured_query.get('original_query', '')).lower()

    modifier_tokens: set[str] = set()
    for m in structured_query.get('modifiers', []):
        modifier_tokens |= tokenize_text(m)

    supporting_tokens: set[str] = set()
    for t in structured_query.get('supporting_terms', []):
        supporting_tokens |= tokenize_text(t)

    return {
        'raw_query': raw_query,
        'modifier_tokens': modifier_tokens,
        'supporting_tokens': supporting_tokens,
    }


def contains_any_phrase(text: str, phrases) -> bool:
    lowered = str(text).lower()
    return any(p in lowered for p in phrases)


def allows_resolved_language(query_ctx: dict[str, Any]) -> bool:
    return contains_any_phrase(query_ctx.get('raw_query', ''), QUERY_EXCEPTION_TERMS)


def allows_causal_language(query_ctx: dict[str, Any]) -> bool:
    return contains_any_phrase(query_ctx.get('raw_query', ''), CAUSAL_TRIGGER_TERMS)


def is_resolved_candidate(candidate: dict[str, Any]) -> bool:
    return 'resolved' in str(candidate.get('term', '')).lower()


def is_causal_variant(candidate: dict[str, Any]) -> bool:
    return contains_any_phrase(str(candidate.get('term', '')), CAUSAL_TRIGGER_TERMS)


def is_pregnancy_variant(candidate: dict[str, Any]) -> bool:
    return contains_any_phrase(str(candidate.get('term', '')), PREGNANCY_TRIGGER_TERMS)


def is_generic_anchor(candidate: dict[str, Any], condition: str) -> bool:
    extra = tokenize_text(candidate.get('term', '')) - tokenize_text(condition) - {'disorder', 'finding'}
    return len(extra) == 0 if tokenize_text(condition) else False


def serialize_candidate_output(row: dict[str, Any], index: int | None = None) -> dict[str, Any]:
    payload = {
        'presentation_score': float(row.get('presentation_score', 0.0)),
        'snomed_code': row['snomed_code'],
        'term': row['term'],
        'semantic_tag': row.get('semantic_tag', ''),
        'confidence_tier': row['confidence_tier'],
        'candidate_role': row.get('candidate_role'),
        'evidence': {
            'in_qof': bool(row['in_qof']),
            'in_opencodelists': bool(row['in_opencodelists']),
            'usage_count_nhs': float(row['usage_count_nhs']),
        },
        'retrieval_features': {
            'fusion_score': float(row.get('fusion_score', 0.0)),
            'rrf_score': float(row.get('rrf_score', row.get('fusion_score', 0.0))),
            'semantic_score_max': float(row.get('semantic_score_max', row.get('semantic_score', 0.0))),
            'bm25_score_max': float(row.get('bm25_score_max', row.get('bm25_score', 0.0))),
            'lexical_overlap_max': float(row.get('lexical_overlap_max', row.get('lexical_overlap', 0.0))),
            'term_precision_max': float(row.get('term_precision_max', row.get('term_precision', 0.0))),
            'specificity_score_max': float(row.get('specificity_score_max', row.get('specificity_score', 0.0))),
            'query_coverage_count': int(row.get('query_coverage_count', 0)),
        },
        'reranker_features': {
            'rerank_score': float(row.get('rerank_score', 0.0)),
            'relevance_score': float(row.get('relevance_score', 0.0)),
            'condition_relevance': row.get('condition_relevance', {}),
            'ce_score': float(row.get('ce_score', 0.0)),
            'ce_score_by_condition': row.get('ce_score_by_condition', {}),
            'matched_conditions_from_ce': row.get('matched_conditions_from_ce', []),
            'dominant_condition_from_ce': row.get('dominant_condition_from_ce'),
        },
        'matched_conditions': row.get('matched_conditions', []),
        'dominant_condition': row.get('dominant_condition', 'unassigned'),
        'ranking_components': row.get('ranking_components', {}),
        'retrieval_method': row.get('retrieval_method', 'direct'),
        'hierarchy_parent_code': row.get('hierarchy_parent_code'),
        'hierarchy_parent_term': row.get('hierarchy_parent_term'),
        'retrieval_trace': row.get('retrieval_trace', []),
    }

    if index is not None:
        payload['presentation_rank'] = index

    return payload

In [144]:
# ── Output Formatter (revised, aligned) ──────────────────────────────────────

BUCKET_ORDER = ['include_candidates', 'review_candidates', 'specific_variants', 'suppressed_candidates']


class LLMExplanationFormatter:
    def __init__(self, config: Any):
        self.config = config

    @staticmethod
    def _deterministic_rationale(candidate: dict[str, Any]) -> str:
        evidence = candidate['evidence']
        role = candidate.get('candidate_role', '')
        ranking = candidate.get('ranking_components', {})

        reasons = []

        # Role-aware explanation aligned with revised pipeline
        if role == 'core':
            reasons.append('broad anchor concept selected after relevance filtering and reranking')
        elif role == 'variant':
            reasons.append('relevant condition-linked concept kept for analyst review')
        elif role == 'specific':
            reasons.append('more specific lower-priority concept retained as depth')
        elif role == 'suppress':
            reasons.append('filtered from first-pass review but retained for traceability')

        # Reranker-aware signal
        ce_score = ranking.get('ce_score')
        rerank_score = ranking.get('rerank_score')

        if ce_score is not None and float(ce_score) > 0:
            reasons.append(f"cross-encoder score {float(ce_score):.3f}")
        elif rerank_score is not None and float(rerank_score) > 0:
            reasons.append(f"rerank score {float(rerank_score):.3f}")

        # Broadness / anchor signal
        anchor_bonus = ranking.get('anchor_bonus')
        if anchor_bonus is not None and float(anchor_bonus) > 0:
            reasons.append('favoured as a broader anchor candidate')

        # Evidence signal
        if evidence['in_qof']:
            reasons.append('QOF-supported code')
        if evidence['in_opencodelists']:
            reasons.append('present in OpenCodelists')
        if evidence['usage_count_nhs'] > 0:
            reasons.append(f"NHS usage count {int(evidence['usage_count_nhs'])}")

        if not reasons:
            reasons.append('retained after relevance filtering with limited supporting evidence')

        return '; '.join(reasons)

    def _serialize_candidate(self, candidate: dict[str, Any]) -> dict[str, Any]:
        return {
            'presentation_score': candidate.get('presentation_score'),
            'snomed_code': candidate['snomed_code'],
            'term': candidate['term'],
            'confidence_tier': candidate['confidence_tier'],
            'candidate_role': candidate.get('candidate_role'),
            'rationale': self._deterministic_rationale(candidate),
            'evidence': {
                'in_qof': candidate['evidence']['in_qof'],
                'in_opencodelists': candidate['evidence']['in_opencodelists'],
                'usage_count_nhs': candidate['evidence']['usage_count_nhs'],
            },
        }

    def format_candidates(
        self,
        user_query: str,
        candidate_groups: dict[str, list[dict[str, Any]]],
    ) -> tuple[dict[str, list[dict[str, Any]]], str, dict[str, Any]]:
        del user_query

        grouped_output = {bucket: [] for bucket in BUCKET_ORDER}
        for bucket in BUCKET_ORDER:
            for c in candidate_groups.get(bucket, []):
                grouped_output[bucket].append(self._serialize_candidate(c))

        debug_info = {
            'formatter_enabled': False,
            'mode': 'deterministic_revised_formatter',
            'schema_validation_error': None,
        }

        return grouped_output, 'deterministic_revised_formatter', debug_info

In [145]:
# ── Audit Logger (CE-reranker aligned) ───────────────────────────────────────
# Source: audit_logger.py (audit writes are non-fatal in Colab)

def generate_run_id() -> str:
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    return f'run_{ts}_{uuid.uuid4().hex[:6]}'


class RunLogger:
    def __init__(
        self,
        config: Any,
        run_id: str,
        original_query: str,
        config_snapshot: dict[str, Any] | None = None,
    ):
        self.config = config
        self.run_id = run_id
        self.original_query = original_query
        self.config_snapshot = config_snapshot or {}
        self.start_time = datetime.now()

        self.trace: dict[str, Any] = {
            'normalized_query': None,
            'structured_query': None,
            'search_jobs': [],
            'retrieval_batches_summary': [],
            'hierarchy_enrichment_summary': [],
            'fused_candidates_preview': [],
            'gate_summary': {},
            'gate_candidates_preview': [],
            'cross_encoder_summary': {},
            'cross_encoder_candidates_preview': [],
            'final_decisions': [],
            'explanation_mode': None,
            'formatter_debug': None,
        }

    def log_normalized_query(self, v: str) -> None:
        self.trace['normalized_query'] = v

    def log_structured_query(self, v: dict[str, Any]) -> None:
        self.trace['structured_query'] = v

    def log_search_jobs(self, v: list) -> None:
        self.trace['search_jobs'] = v

    def log_final_decisions(self, v: Any) -> None:
        self.trace['final_decisions'] = v

    def log_explanation_mode(self, v: str) -> None:
        self.trace['explanation_mode'] = v

    def log_formatter_debug(self, v: dict) -> None:
        self.trace['formatter_debug'] = v

    def log_retrieval_batches(self, search_jobs: list, retrieval_batches: list) -> None:
        self.trace['retrieval_batches_summary'] = [
            {
                'query_text': job['query_text'],
                'query_type': job['query_type'],
                'clinical_focus': job.get('clinical_focus'),
                'result_count': len(batch),
            }
            for job, batch in zip(search_jobs, retrieval_batches)
        ]

    def log_hierarchy_enrichment(self, enriched_batches: list[list[dict[str, Any]]]) -> None:
        self.trace['hierarchy_enrichment_summary'] = [
            {
                'batch_index': i,
                'total_candidates_after_enrichment': len(batch),
                'hierarchy_added_count': sum(
                    1 for c in batch if c.get('retrieval_method') == 'hierarchy_child'
                ),
            }
            for i, batch in enumerate(enriched_batches)
        ]

    def log_fused_candidates(self, fused_candidates: list[dict[str, Any]]) -> None:
        self.trace['fused_candidates_preview'] = [
            {
                'snomed_code': c['snomed_code'],
                'term': c['term'],
                'fusion_score': c.get('fusion_score', 0.0),
                'rrf_score': c.get('rrf_score', 0.0),
                'query_types_hit': c.get('query_types_hit', []),
                'clinical_focuses_hit': c.get('clinical_focuses_hit', []),
                'query_coverage_count': c.get('query_coverage_count', 0),
                'retrieval_method': c.get('retrieval_method', 'direct'),
            }
            for c in sorted(fused_candidates, key=lambda x: x.get('fusion_score', 0.0), reverse=True)[:10]
        ]

    def log_gate_candidates(
        self,
        gate_candidates: list[dict[str, Any]],
        relevance_threshold: float | None = None,
        top_n_per_condition: int | None = None,
    ) -> None:
        self.trace['gate_summary'] = {
            'relevance_threshold': relevance_threshold,
            'top_n_per_condition': top_n_per_condition,
            'survivor_count': len(gate_candidates),
        }

        self.trace['gate_candidates_preview'] = [
            {
                'snomed_code': c['snomed_code'],
                'term': c['term'],
                'rerank_score': c.get('rerank_score', 0.0),
                'relevance_score': c.get('relevance_score', 0.0),
                'matched_conditions_from_gate': c.get('matched_conditions_from_gate', []),
                'dominant_condition_from_gate': c.get('dominant_condition_from_gate'),
                'retrieval_method': c.get('retrieval_method', 'direct'),
            }
            for c in sorted(gate_candidates, key=lambda x: x.get('rerank_score', 0.0), reverse=True)[:20]
        ]

    def log_cross_encoder_candidates(
        self,
        ce_candidates: list[dict[str, Any]],
        model_name: str | None = None,
        top_n_per_condition: int | None = None,
        ce_condition_weight: float | None = None,
        ce_query_weight: float | None = None,
    ) -> None:
        self.trace['cross_encoder_summary'] = {
            'model_name': model_name,
            'top_n_per_condition': top_n_per_condition,
            'ce_condition_weight': ce_condition_weight,
            'ce_query_weight': ce_query_weight,
            'survivor_count': len(ce_candidates),
        }

        self.trace['cross_encoder_candidates_preview'] = [
            {
                'snomed_code': c['snomed_code'],
                'term': c['term'],
                'ce_score': c.get('ce_score', 0.0),
                'rerank_score': c.get('rerank_score', 0.0),
                'relevance_score': c.get('relevance_score', 0.0),
                'matched_conditions_from_gate': c.get('matched_conditions_from_gate', []),
                'dominant_condition_from_gate': c.get('dominant_condition_from_gate'),
                'matched_conditions_from_ce': c.get('matched_conditions_from_ce', []),
                'dominant_condition_from_ce': c.get('dominant_condition_from_ce'),
                'ce_score_by_condition': c.get('ce_score_by_condition', {}),
                'retrieval_method': c.get('retrieval_method', 'direct'),
            }
            for c in sorted(ce_candidates, key=lambda x: x.get('ce_score', 0.0), reverse=True)[:20]
        ]

    def finish(self, final_structured_output: Any) -> Any:
        packet = {
            'run_id': self.run_id,
            'timestamp': self.start_time.isoformat(),
            'duration_seconds': (datetime.now() - self.start_time).total_seconds(),
            'query': self.original_query,
            'config_snapshot': self.config_snapshot,
            'trace': self.trace,
            'final_structured_output': final_structured_output,
        }

        try:
            audit_dir = Path(self.config.base_dir) / 'audit'
            audit_dir.mkdir(parents=True, exist_ok=True)
            audit_path = audit_dir / f'{self.run_id}.json'
            with open(audit_path, 'w', encoding='utf-8') as fh:
                json.dump(packet, fh, indent=2)
            logger.info('Audit written to %s', audit_path)
        except Exception as exc:
            logger.warning('Audit write skipped (non-fatal): %s', exc)

        return final_structured_output

In [146]:
# ── Config (CE-reranker aligned) ──────────────────────────────────────────────
# Source: config.py (adapted — reads directly from notebook variables)

@dataclass(frozen=True)
class Config:
    base_dir:               Path
    snomed_path:            Path
    edge_path:              Path
    chroma_persist_dir:     Path
    embeddings_dir:         Path

    demo_query:             str = 'Obesity, diabetes mellitus, and hypertension'
    top_k:                  int = 10

    chroma_collection_name: str = 'snomed_master_v4_retrieval'
    embedding_model_name:   str = 'BAAI/bge-small-en'
    llm_model:              str = 'llama3.1'

    # New real reranker config
    cross_encoder_model_name: str = 'BAAI/bge-reranker-v2-m3'
    gate_top_n_per_condition: int = 50
    ce_top_n_per_condition:   int = 20
    ce_condition_weight:      float = 0.75
    ce_query_weight:          float = 0.25

    default_top_k:          int = 20
    audit_verbosity:        str = 'standard'

    # Planner support metadata only
    query_type_weights: dict[str, float] = field(default_factory=lambda: {
        'primary_condition': 1.0,
        'secondary_condition': 1.0,
        'modifier': 0.5,
        'supporting_term': 0.35,
        'combined': 0.0,
    })

    def to_snapshot(self, user_query: str, effective_top_k: int) -> dict[str, Any]:
        return {
            'query': user_query,
            'input_assets': {
                'snomed_csv': str(self.snomed_path),
                'hierarchy_edge_csv': str(self.edge_path),
                'chroma_persist_dir': str(self.chroma_persist_dir),
                'chroma_collection_name': self.chroma_collection_name,
            },
            'models': {
                'embedding_model_name': self.embedding_model_name,
                'llm_model': self.llm_model,
                'cross_encoder_model_name': self.cross_encoder_model_name,
            },
            'runtime': {
                'effective_top_k': effective_top_k,
                'default_top_k': self.default_top_k,
                'audit_verbosity': self.audit_verbosity,
                'gate_top_n_per_condition': self.gate_top_n_per_condition,
                'ce_top_n_per_condition': self.ce_top_n_per_condition,
                'ce_condition_weight': self.ce_condition_weight,
                'ce_query_weight': self.ce_query_weight,
            },
            'architecture': {
                'condition_isolated_planning': True,
                'hierarchy_zero_trust_enrichment': True,
                'rrf_fusion': True,
                'relevance_gate_reranker': True,
                'cross_encoder_reranker': True,
                'bounded_authority_decisioning': True,
            },
        }


def build_config_from_notebook() -> Config:
    """Build Config from the notebook settings above."""
    snomed_path = Path(SNOMED_PATH)

    return Config(
        base_dir=snomed_path.parent,
        snomed_path=snomed_path,
        edge_path=Path(EDGE_PATH),
        chroma_persist_dir=Path(CHROMA_DIR),
        embeddings_dir=Path(EMBEDDINGS_DIR),
        demo_query=DEMO_QUERY,
        top_k=TOP_K,
        chroma_collection_name='snomed_master_v4_retrieval',
        embedding_model_name=EMBEDDING_MODEL_NAME,
        llm_model=LLM_MODEL,
        cross_encoder_model_name='BAAI/bge-reranker-v2-m3',
        gate_top_n_per_condition=50,
        ce_top_n_per_condition=20,
        ce_condition_weight=0.75,
        ce_query_weight=0.25,
    )

---
## Section 4 · Pipeline Orchestration

`run_pipeline()` wires together all stages above.  
It mirrors `main.py` exactly, using `build_config_from_notebook()` instead of `build_config()`.

In [147]:
# ── Pipeline Runner (final, corrected) ───────────────────────────────────────

def run_pipeline(user_query: str, top_k: int | None = None) -> Any:
    config = build_config_from_notebook()
    effective_top_k = top_k or config.default_top_k
    run_id = generate_run_id()

    tracker = RunLogger(
        config,
        run_id,
        user_query,
        config_snapshot=config.to_snapshot(user_query, effective_top_k),
    )

    normalized_query = normalize_query(user_query)
    tracker.log_normalized_query(normalized_query)

    data_loader = DataLoader(config)
    decomposer = QueryDecomposer(config.llm_model)
    planner = QueryPlanner(config.query_type_weights)
    retriever = HybridRetriever(config, data_loader)
    hierarchy_enricher = HierarchyEnricher(config.snomed_path, config.edge_path)
    fusion_engine = CandidateFusionEngine()

    # ── Stage 1: Relevance Gate ──────────────────────────────────────────────
    relevance_reranker = RelevanceGateReranker(
        relevance_threshold=0.10,
        top_n_per_condition=config.gate_top_n_per_condition,
    )

    # ── Stage 2: Cross-Encoder Reranker ──────────────────────────────────────
    cross_encoder_reranker = CrossEncoderReranker(
        model_name=config.cross_encoder_model_name,
        top_n_per_condition=config.ce_top_n_per_condition,
        batch_size=16,
        max_length=512,
    )

    decision_engine = DecisionEngine()
    formatter = LLMExplanationFormatter(config)

    # 1. Query decomposition
    structured_query = decomposer.decompose(normalized_query)
    tracker.log_structured_query(structured_query)

    # 2. Build search jobs
    search_jobs = planner.build_search_queries(structured_query)
    tracker.log_search_jobs(search_jobs)

    # 3. Retrieval
    retrieval_batches = [
        retriever.retrieve(job, top_k=effective_top_k)
        for job in search_jobs
    ]
    tracker.log_retrieval_batches(search_jobs, retrieval_batches)

    # 4. Hierarchy enrichment
    enriched_batches = [
        hierarchy_enricher.enrich_batch(batch)
        for batch in retrieval_batches
    ]
    tracker.log_hierarchy_enrichment(enriched_batches)

    # 5. Fusion
    fused_candidates = fusion_engine.fuse(enriched_batches)
    tracker.log_fused_candidates(fused_candidates)

    # 6. Relevance Gate
    gate_candidates = relevance_reranker.rerank(
        fused_candidates,
        structured_query=structured_query,
    )
    tracker.log_gate_candidates(
        gate_candidates,
        relevance_threshold=relevance_reranker.relevance_threshold,
        top_n_per_condition=relevance_reranker.top_n_per_condition,
    )

    # 7. Cross-Encoder Rerank (STRICT per-condition only)
    ce_reranked_candidates = cross_encoder_reranker.rerank(
        gate_candidates,
        structured_query=structured_query,
    )
    tracker.log_cross_encoder_candidates(
        ce_reranked_candidates,
        model_name=cross_encoder_reranker.model_name,
        top_n_per_condition=cross_encoder_reranker.top_n_per_condition,
    )

    # 8. Decision Engine
    candidate_groups = decision_engine.assign_final_decisions(
        ce_reranked_candidates,
        structured_query=structured_query,
        top_k=effective_top_k,
    )
    tracker.log_final_decisions(candidate_groups)

    # 9. Formatter
    formatted_output, explanation_mode, formatter_debug = formatter.format_candidates(
        normalized_query,
        candidate_groups,
    )
    tracker.log_explanation_mode(explanation_mode)
    tracker.log_formatter_debug(formatter_debug)

    return tracker.finish(formatted_output)

---
## Section 5 · Run Demo Query

Runs the pipeline with `DEMO_QUERY` and `TOP_K` from Section 2.  
Edit those variables and re-run this cell to try a different query.

In [148]:
pipeline_output = run_pipeline(DEMO_QUERY, top_k=TOP_K)
print(json.dumps(pipeline_output, indent=2))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

{
  "include_candidates": [
    {
      "presentation_score": 0.616061,
      "snomed_code": "414916001",
      "term": "Obesity (disorder)",
      "confidence_tier": "MEDIUM",
      "candidate_role": "core",
      "rationale": "broad anchor concept selected after relevance filtering and reranking; cross-encoder score 0.968; NHS usage count 265430",
      "evidence": {
        "in_qof": false,
        "in_opencodelists": false,
        "usage_count_nhs": 265430.0
      }
    },
    {
      "presentation_score": 0.750639,
      "snomed_code": "73211009",
      "term": "Diabetes mellitus (disorder)",
      "confidence_tier": "HIGH",
      "candidate_role": "core",
      "rationale": "broad anchor concept selected after relevance filtering and reranking; cross-encoder score 0.969; QOF-supported code; present in OpenCodelists; NHS usage count 249270",
      "evidence": {
        "in_qof": true,
        "in_opencodelists": true,
        "usage_count_nhs": 249270.0
      }
    },
    {
     

In [149]:
conversational_query = 'A patient with poorly controlled type 2 diabetes, presenting with morbid obesity and also history of hypertension'

# This will specifically show how the LLM decomposes the string
config = build_config_from_notebook()
decomposer = QueryDecomposer(config.llm_model)
structured_data = decomposer.decompose(conversational_query)

print('--- LLM Decomposition Result ---')
import json
print(json.dumps(structured_data, indent=2))

--- LLM Decomposition Result ---
{
  "original_query": "A patient with poorly controlled type 2 diabetes, presenting with morbid obesity and also history of hypertension",
  "primary_condition": "Type 2 Diabetes Mellitus",
  "secondary_conditions": [
    "Morbid Obesity",
    "Hypertension"
  ],
  "modifiers": [],
  "supporting_terms": []
}


In [150]:
pipeline_output = run_pipeline(conversational_query, top_k=TOP_K)
print(json.dumps(pipeline_output, indent=2))

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

{
  "include_candidates": [
    {
      "presentation_score": 0.763295,
      "snomed_code": "44054006",
      "term": "Diabetes mellitus type 2 (disorder)",
      "confidence_tier": "HIGH",
      "candidate_role": "core",
      "rationale": "broad anchor concept selected after relevance filtering and reranking; cross-encoder score 0.993; QOF-supported code; present in OpenCodelists; NHS usage count 4666570",
      "evidence": {
        "in_qof": true,
        "in_opencodelists": true,
        "usage_count_nhs": 4666570.0
      }
    },
    {
      "presentation_score": 0.636656,
      "snomed_code": "238136002",
      "term": "Morbid obesity (disorder)",
      "confidence_tier": "MEDIUM",
      "candidate_role": "core",
      "rationale": "broad anchor concept selected after relevance filtering and reranking; cross-encoder score 0.997; NHS usage count 32550",
      "evidence": {
        "in_qof": false,
        "in_opencodelists": false,
        "usage_count_nhs": 32550.0
      }
    

---
## Section 6 · Optional Extension Hooks

Use this area to connect evaluation tooling, batch runners, or widget UIs.  
`run_pipeline()` returns a structured dict — pass it to whatever you build here.

In [152]:
"""# ── Build / Rebuild Chroma from SNOMED Master v4 ─────────────────────────────
# Creates a fresh Chroma collection from the current SNOMED master table.

import shutil
import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer
from pathlib import Path

# Load concept table
df = pd.read_csv(SNOMED_PATH, dtype=str)
df = df.dropna(subset=['snomed_code', 'term']).copy()
df['snomed_code'] = df['snomed_code'].astype(str).str.strip()
df['term'] = df['term'].astype(str).str.strip()
df['semantic_tag'] = df.get('semantic_tag', '').fillna('').astype(str)
df['opencodelist_clinical_areas'] = df.get('opencodelist_clinical_areas', '').fillna('').astype(str)
df['qof_cluster_description'] = df.get('qof_cluster_description', '').fillna('').astype(str)

# Keep embedding text simple
df['text_for_embedding'] = (
    df['term'] + ' | ' +
    df['semantic_tag'] + ' | ' +
    df['opencodelist_clinical_areas'] + ' | ' +
    df['qof_cluster_description']
).str.strip()

df = df.drop_duplicates(subset=['snomed_code']).copy()

print(f"Loaded {len(df)} concepts from {SNOMED_PATH}")

# Fresh Chroma directory
chroma_path = Path(CHROMA_DIR)
if chroma_path.exists():
    shutil.rmtree(chroma_path)
chroma_path.mkdir(parents=True, exist_ok=True)

# Embedding model
embedder = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    cache_folder=EMBEDDINGS_DIR
)

# Chroma client + collection
client = chromadb.PersistentClient(path=str(chroma_path))
collection = client.get_or_create_collection(name='snomed_master_v4_retrieval')

# Build embeddings
texts = df['text_for_embedding'].tolist()
ids = df['snomed_code'].tolist()

metadatas = [
    {
        'term': str(row.term),
        'semantic_tag': str(row.semantic_tag),
        'in_qof': str(row.in_qof) if 'in_qof' in df.columns else 'False',
        'in_opencodelists': str(row.in_opencodelists) if 'in_opencodelists' in df.columns else 'False',
        'usage_count_nhs': str(row.usage_count_nhs) if 'usage_count_nhs' in df.columns else '0',
        'num_parents': str(row.num_parents) if 'num_parents' in df.columns else '0',
        'num_children': str(row.num_children) if 'num_children' in df.columns else '0',
    }
    for row in df.itertuples(index=False)
]

batch_size = 1000
for start in range(0, len(df), batch_size):
    end = min(start + batch_size, len(df))
    batch_ids = ids[start:end]
    batch_texts = texts[start:end]
    batch_meta = metadatas[start:end]
    batch_embeddings = embedder.encode(batch_texts, show_progress_bar=False).tolist()

    collection.add(
        ids=batch_ids,
        documents=batch_texts,
        metadatas=batch_meta,
        embeddings=batch_embeddings
    )

    print(f"Added {end}/{len(df)}")

print("\nChroma rebuild complete.")
print("Collection name: snomed_master_v4_retrieval")
print(f"Persisted at: {CHROMA_DIR}")
print("Total records in collection:", collection.count())

"""

'# ── Build / Rebuild Chroma from SNOMED Master v4 ─────────────────────────────\n# Creates a fresh Chroma collection from the current SNOMED master table.\n\nimport shutil\nimport chromadb\nimport pandas as pd\nfrom sentence_transformers import SentenceTransformer\nfrom pathlib import Path\n\n# Load concept table\ndf = pd.read_csv(SNOMED_PATH, dtype=str)\ndf = df.dropna(subset=[\'snomed_code\', \'term\']).copy()\ndf[\'snomed_code\'] = df[\'snomed_code\'].astype(str).str.strip()\ndf[\'term\'] = df[\'term\'].astype(str).str.strip()\ndf[\'semantic_tag\'] = df.get(\'semantic_tag\', \'\').fillna(\'\').astype(str)\ndf[\'opencodelist_clinical_areas\'] = df.get(\'opencodelist_clinical_areas\', \'\').fillna(\'\').astype(str)\ndf[\'qof_cluster_description\'] = df.get(\'qof_cluster_description\', \'\').fillna(\'\').astype(str)\n\n# Keep embedding text simple\ndf[\'text_for_embedding\'] = (\n    df[\'term\'] + \' | \' +\n    df[\'semantic_tag\'] + \' | \' +\n    df[\'opencodelist_clinical_areas\'